# PZ data challenge: task set 3

**Task set 3: non-representative training, plus COSMOS2020 many-band photo-z.** The reference redshift is:
- the spectroscopic redshift (`redshift`) for objects selected by a spectroscopic survey;
- `redshift_manyband` for objects selected only by COSMOS2020, whose `redshift` is NaN.

The COSMOS-only objects are the faint, test-like part of the training set, but they can only be scored against their many-band photo-z, so their metrics include the label noise.

This notebook only trains. For each of the four configurations of the task set (cardinal and flagship, 1yr and 10yr) it trains the model on the training file, holding out `VAL_FRAC` of it (split by `SEED`), and saves it as a single `.pkl` under its official name in `OUT_DIR` (the subtask 2 file, via `znn.save_model_file`), shared by all four task-set notebooks. Everything that uses the models, the test-set p(z)s and the scores on the held-out objects, is in `pz_challenge_results.ipynb`, which predicts from these files.

The model is trained with `znn.train_model`, on the photometry as given. It is a pair of `znn.build_model` networks (residual squeeze-excitation nets), after the moment networks of Jeffrey & Wandelt 2020: F predicts log(1 + z), then G, trained on F's residuals with F fixed, predicts their scale, as a Student-t with `NU` degrees of freedom. The p(z) (in `pz_challenge_results.ipynb`) is that Student-t around F's prediction. With `PRETRAINING = "popcosmos"`, F starts from its pop-cosmos pre-trained weights shipped with znn, every layer trainable; G always starts from scratch. A configuration whose model file is already in `OUT_DIR` is not retrained; delete the file to retrain.

Run this from the repo root with the `qp` conda env as the kernel. The filter curves are the ones shipped with RAIL, which `pz_data_challenge` installs.

## Configuration

In [1]:
from rail.utils.path_utils import find_rail_file

import znn  # run from the repo root so this picks up the local source

TASKSET = 3
SIMS = ["cardinal", "flagship"]
SCENARIOS = ["1yr", "10yr"]

DATA_DIR = "../pz_data_challenge/scripts/public"
# Starting point of F: "popcosmos" (the pre-trained weights shipped with znn, nothing frozen) or None (from scratch).
# Each choice writes to its own directory, so runs with and without pre-training never overwrite each other.
PRETRAINING = None
NU = znn.DEFAULT_NU  # degrees of freedom of the Student-t G learns
# Method version, appended to OUT_DIR so runs of different versions never overwrite each other ("" for none). It
# starts with "build_model_res_se_log1pz", the name the residual SE model (znn.build_model) had when it was added,
# which keeps its runs apart from the old model's, then the Student-t moment network and its NU
RUN = f"build_model_res_se_log1pz_moment_t{NU:g}"
# where the runs are saved (models, submission files, cached predictions); run_challenge.py sets it to its
# --output-root
OUTPUT_ROOT = "outputs"
OUT_DIR = (f"{OUTPUT_ROOT}/znn" if PRETRAINING is None else f"{OUTPUT_ROOT}/znn_pretrained_{PRETRAINING}") + (f"_{RUN}" if RUN else "")  # submission files for all task sets; the contents of the submission tarball

# Filter curves shipped with RAIL; the challenge's catalogs.yaml ("cardinal_roman_rubin") maps the
# catalogue columns to these DC2LSST_* and roman_* filters
FILTER_DIR = find_rail_file("examples_data/estimation_data/data/FILTER")

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength
BANDS = {
    "u": ("mag_u_lsst", f"{FILTER_DIR}/DC2LSST_u.res"),
    "g": ("mag_g_lsst", f"{FILTER_DIR}/DC2LSST_g.res"),
    "r": ("mag_r_lsst", f"{FILTER_DIR}/DC2LSST_r.res"),
    "i": ("mag_i_lsst", f"{FILTER_DIR}/DC2LSST_i.res"),
    "z": ("mag_z_lsst", f"{FILTER_DIR}/DC2LSST_z.res"),
    "y": ("mag_y_lsst", f"{FILTER_DIR}/DC2LSST_y.res"),
    "Y": ("mag_Y_roman", f"{FILTER_DIR}/roman_Y106.res"),
    "J": ("mag_J_roman", f"{FILTER_DIR}/roman_J129.res"),
    "H": ("mag_H_roman", f"{FILTER_DIR}/roman_H158.res"),
}
REF_BAND = "i"  # magnitudes are normalized by this band
FEATURE_CONFIG = znn.DEFAULT_FEATURE_CONFIG

VAL_FRAC = 0.2  # fraction of the training set held out for evaluation (in pz_challenge_results.ipynb, which must match)
EPOCHS = 100  # max epochs of F and of G (early stopping usually ends sooner)
SEED = 42

I0000 00:00:1793228868.977311  547288 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os

import numpy as np
from sklearn.model_selection import train_test_split

os.makedirs(OUT_DIR, exist_ok=True)
print("znn from", znn.__file__)

znn from /global/u2/j/jaimerz/UCL/mlpz_cnn_deepset/znn/__init__.py


## Reference redshifts

In [3]:
SPEC_SURVEYS = ["DEEP2_LSST", "DESI_BGS", "DESI_ELG_LOP", "DESI_LRG", "VVDSf02", "zCOSMOS"]


def reference_redshifts(train):
    """Reference redshift per training object, and the held-out subsets to score."""
    is_spec = train[[c for c in SPEC_SURVEYS if c in train]].astype(bool).any(axis=1).to_numpy()
    is_cosmos = train["COSMOS"].astype(bool).to_numpy()
    is_cosmos_only = is_cosmos & ~is_spec
    z_spec = train["redshift"].to_numpy()
    z_manyband = train["redshift_manyband"].to_numpy()
    assert np.isnan(z_spec[is_cosmos_only]).all() and np.isfinite(z_spec[is_spec]).all()

    z_ref = np.where(is_cosmos_only, z_manyband, z_spec)
    assert np.isfinite(z_ref).all(), "every training object should have a reference redshift"

    # How noisy are the many-band labels? Measured on objects with both a spec-z and a many-band photo-z,
    # which are bright, so this understates the noise at the faint end.
    overlap = is_cosmos & is_spec
    dz = (z_manyband[overlap] - z_spec[overlap]) / (1 + z_spec[overlap])
    bias, _, scatter, _, outliers = znn.get_biweight_mean_sigma_outlier(dz)
    print(
        f"spec-selected: {is_spec.mean():.1%}, COSMOS-only: {is_cosmos_only.mean():.1%}; many-band vs spec-z on "
        f"{overlap.sum()} overlap objects: bias {bias:.4f}, scatter {scatter:.4f}, |dz|>0.2 rate {outliers:.4f}"
    )

    subsets = {
        "held-out spec-selected (vs spec-z)": is_spec,
        "held-out COSMOS-only (vs many-band photo-z)": is_cosmos_only,
        "held-out all": np.ones(len(train), dtype=bool),
    }
    return z_ref, subsets

## Run all configurations

For each configuration whose model file is not in `OUT_DIR` yet: train on the fit split and save the model file. The held-out split is left out of training so `pz_challenge_results.ipynb` can score on it.

In [4]:
for SIM in SIMS:
    for SCENARIO in SCENARIOS:
        config = f"{SIM}_{SCENARIO}"
        print(f"\n========== task set {TASKSET}: {config} ==========")
        model_file = os.path.join(OUT_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_pz_model_{SCENARIO}.pkl")
        if os.path.exists(model_file):
            print("found", model_file, "(delete it to retrain)")
            continue

        train_file = os.path.join(DATA_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_training_{SCENARIO}.hdf5")
        train = znn.read_catalog(train_file)
        z_ref, _ = reference_redshifts(train)

        # the held-out split is not used here, only kept out of training (pz_challenge_results.ipynb scores on it)
        idx_fit, idx_val = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED)
        model, _ = znn.train_model(
            znn.build_model, train.iloc[idx_fit], BANDS, REF_BAND, FEATURE_CONFIG, Y=z_ref[idx_fit],
            nu=NU, EPOCHS=EPOCHS, random_state=SEED, pretraining=PRETRAINING,
        )

        # subtask 2 submission file: the trained model
        znn.save_model_file(model_file, model, BANDS, REF_BAND, FEATURE_CONFIG)
        print("saved", model_file)


========== task set 3: cardinal_1yr ==========
spec-selected: 64.9%, COSMOS-only: 35.1%; many-band vs spec-z on 4284 overlap objects: bias 0.0003, scatter 0.0135, |dz|>0.2 rate 0.0047


I0000 00:00:1793228945.751526  547288 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38478 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:03:00.0, compute capability: 8.0
I0000 00:00:1793228945.755902  547288 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 38478 MB memory:  -> device: 1, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:41:00.0, compute capability: 8.0
I0000 00:00:1793228945.760397  547288 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 38478 MB memory:  -> device: 2, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:82:00.0, compute capability: 8.0
I0000 00:00:1793228945.764672  547288 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 38478 MB memory:  -> device: 3, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:c1:00.0, compute capability: 8.0



=== Base model, on all objects with the photometry as given ===
Epoch 1/100


I0000 00:00:1793228949.357806  549066 service.cc:153] XLA service 0x7f1e5c037a50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793228949.357828  549066 service.cc:161]   StreamExecutor [0]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793228949.357836  549066 service.cc:161]   StreamExecutor [1]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793228949.357839  549066 service.cc:161]   StreamExecutor [2]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793228949.357842  549066 service.cc:161]   StreamExecutor [3]: NVIDIA A100-SXM4-40GB, Compute Capability 8.0 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793228949.391798  549066 dump_mlir_util.cc:269] disabling MLIR crash repro

250/250 - 31s - 123ms/step - loss: 0.0073 - val_loss: 0.0051 - learning_rate: 0.0010
Epoch 2/100
250/250 - 1s - 3ms/step - loss: 0.0058 - val_loss: 0.0050 - learning_rate: 0.0010
Epoch 3/100
250/250 - 1s - 3ms/step - loss: 0.0055 - val_loss: 0.0046 - learning_rate: 0.0010
Epoch 4/100
250/250 - 1s - 3ms/step - loss: 0.0053 - val_loss: 0.0044 - learning_rate: 0.0010
Epoch 5/100
250/250 - 1s - 3ms/step - loss: 0.0052 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 6/100
250/250 - 1s - 3ms/step - loss: 0.0051 - val_loss: 0.0044 - learning_rate: 0.0010
Epoch 7/100
250/250 - 1s - 3ms/step - loss: 0.0050 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 8/100
250/250 - 1s - 3ms/step - loss: 0.0049 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 9/100
250/250 - 1s - 3ms/step - loss: 0.0048 - val_loss: 0.0044 - learning_rate: 0.0010
Epoch 10/100

Epoch 10: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
250/250 - 1s - 3ms/step - loss: 0.0048 - val_loss: 0.0044 - learning_rat

/global/u2/j/jaimerz/UCL/mlpz_cnn_deepset/znn/data.py:158: RuntimeWarning: invalid value encountered in subtract
  mags_normed = mags - mag_i[None, :]  # normalize by i-band mag, per band, before binning


... (89 outputs left out)

282/282 - 7s - 25ms/step - loss: 0.0068 - val_loss: 0.0065
Epoch 2/10
282/282 - 1s - 3ms/step - loss: 0.0067 - val_loss: 0.0065
Epoch 3/10
282/282 - 1s - 3ms/step - loss: 0.0066 - val_loss: 0.0065
Epoch 4/10
282/282 - 1s - 3ms/step - loss: 0.0066 - val_loss: 0.0065
Epoch 5/10
282/282 - 1s - 3ms/step - loss: 0.0065 - val_loss: 0.0065
Epoch 6/10
282/282 - 1s - 3ms/step - loss: 0.0064 - val_loss: 0.0066
Epoch 7/10
282/282 - 1s - 3ms/step - loss: 0.0064 - val_loss: 0.0066
Epoch 8/10
282/282 - 1s - 3ms/step - loss: 0.0063 - val_loss: 0.0066
Epoch 9/10
282/282 - 1s - 3ms/step - loss: 0.0063 - val_loss: 0.0065
Epoch 10/10
282/282 - 1s - 3ms/step - loss: 0.0063 - val_loss: 0.0066

--- Member 10/10 (its own noise realization) ---
Epoch 1/10


I0000 00:00:1793229722.416333  549066 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_575605__.38
I0000 00:00:1793229724.740362  549064 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_575605__.38


282/282 - 7s - 25ms/step - loss: 0.0070 - val_loss: 0.0045
Epoch 2/10
282/282 - 1s - 3ms/step - loss: 0.0069 - val_loss: 0.0045
Epoch 3/10
282/282 - 1s - 3ms/step - loss: 0.0068 - val_loss: 0.0045
Epoch 4/10
282/282 - 1s - 3ms/step - loss: 0.0067 - val_loss: 0.0045
Epoch 5/10
282/282 - 1s - 3ms/step - loss: 0.0066 - val_loss: 0.0046
Epoch 6/10
282/282 - 1s - 3ms/step - loss: 0.0066 - val_loss: 0.0045
Epoch 7/10
282/282 - 1s - 3ms/step - loss: 0.0065 - val_loss: 0.0045
Epoch 8/10
282/282 - 1s - 3ms/step - loss: 0.0065 - val_loss: 0.0046
Epoch 9/10
282/282 - 1s - 3ms/step - loss: 0.0065 - val_loss: 0.0045
Epoch 10/10
282/282 - 1s - 3ms/step - loss: 0.0064 - val_loss: 0.0046


saved outputs/znn_pretrained_popcosmos_noisy_kfold_frombase/pz_challenge_taskset_3_flagship_pz_model_10yr.pkl
